# Nainital Tourism Vulnerability — Data Cleaning & EDA

Cleans and explores the 5 raw survey files under `survey_data/` (Enterprise, Location site-audit,
Tourist, Local Residents, Workers) so the composite-index pipeline described in `workflow.md` and
`VARIABLES_AND_DATA_SOURCES.md` has trustworthy inputs.

**What this notebook does:**
1. Loads each survey and profiles missingness/shape.
2. Harmonizes KoBoToolbox form-versioning artifacts — duplicate/reworded columns and
   label/answer-pair columns — verified by inspection, not blind regex (see the coalesce calls
   below; each maps real column names found in the raw files).
3. Drops columns that are 100% empty (dead form-revision fields nobody ever answered) and flags/nulls
   GPS coordinates outside a Nainital bounding box.
4. Assigns GPS-tagged Enterprise/Resident records to the nearest of the 10 surveyed zones
   (haversine distance, 500m cutoff) — Tourist/Worker surveys aren't GPS-tagged, so they stay
   town-wide only.
5. Runs EDA per survey (distributions, value counts) and a couple of research-benchmark sanity
   checks (see `RESEARCH_BENCHMARKS.md` for full citations).
6. Saves cleaned tables to `../processed/*.csv` and prints a data-quality summary — what got kept,
   what got skipped, and why.

**Rule for "does this line up?"**: every drop/skip decision below is logged and printed, not silent.
If a column is 100% empty, an outlier is out of the Nainital bounding box, or a respondent-level
oddity shows up in EDA, it's called out explicitly rather than quietly excluded.


In [1]:
%pip install -q pandas numpy matplotlib openpyxl



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /private/tmp/claude-501/-Users-shreyitsinha-Documents-Tiss-my-Sem-IV-UFW-dashboard-vulnarability-tourism/40c38ef8-42c9-4173-a0e2-b24ef947bb87/scratchpad/venv/bin/python3.14 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

DATA_DIR = Path("../survey_data")
OUT_DIR = Path("../processed")
OUT_DIR.mkdir(exist_ok=True)

# Nainital town bounding box (generous ~15km x 15km around 29.39N, 79.46E) — anything outside this
# is almost certainly a bad GPS tap, not a real survey location.
NAINITAL_BBOX = dict(lat_min=29.30, lat_max=29.45, lon_min=79.40, lon_max=79.55)


## Research-backed reference numbers

Pulled from `RESEARCH_BENCHMARKS.md` — only the ones with a verified, directly-applicable number are
kept here as constants; everything context-only (Falkenmark, WHO litres/day, waste/tourist) is
documented in that file but **not** hardcoded here, since applying them would need data we don't
have (see the file for why).


In [3]:
BENCHMARKS = {
    "sphere_max_users_per_toilet": {
        "value": 20,
        "note": "Sphere Standards emergency WASH minimum (<20 users/toilet). A humanitarian floor, "
                "not a tourism-comfort standard — use as a 'not in crisis' check, not an adequacy target.",
        "source": "WHO/Sphere Handbook — WASH minimum standards",
    },
    "sphere_min_litres_per_person_day": {
        "value": 15,
        "note": "Sphere emergency minimum water allowance (7.5L/day acceptable briefly in acute drought).",
        "source": "WHO — Technical notes on drinking-water in emergencies",
    },
    "nainital_road_capacity_vehicles": {
        "value": 3000,
        "note": "Approx. vehicle capacity of Nainital's road network.",
        "source": "Himalayan Geographic / Vision IAS coverage of Uttarakhand NGT-directed TCC assessment (2025)",
    },
    "nainital_peak_daily_vehicles": {
        "value": (3000, 4000),
        "note": "Reported daily tourist-vehicle inflow during peak season (range).",
        "source": "Himalayan Geographic — Himalayan overtourism crisis",
    },
    "nainital_peak_weekend_vehicles": {
        "value": 50000,
        "note": "Reported peak-summer-weekend vehicle attempts vs ~3,000 capacity — i.e. ~16.7x capacity.",
        "source": "Himalayan Geographic — Himalayan overtourism crisis",
    },
    "india_msw_avg_kg_per_person_day": {
        "value": 0.57,
        "note": "National average solid-waste generation; tourist hill towns run above this "
                "(qualitative — no verified per-tourist Nainital figure exists, see RESEARCH_BENCHMARKS.md).",
        "source": "NITI Aayog — Waste-Wise Cities",
    },
}

for k, v in BENCHMARKS.items():
    print(f"- {k}: {v['value']}  [{v['source']}]")


- sphere_max_users_per_toilet: 20  [WHO/Sphere Handbook — WASH minimum standards]
- sphere_min_litres_per_person_day: 15  [WHO — Technical notes on drinking-water in emergencies]
- nainital_road_capacity_vehicles: 3000  [Himalayan Geographic / Vision IAS coverage of Uttarakhand NGT-directed TCC assessment (2025)]
- nainital_peak_daily_vehicles: (3000, 4000)  [Himalayan Geographic — Himalayan overtourism crisis]
- nainital_peak_weekend_vehicles: 50000  [Himalayan Geographic — Himalayan overtourism crisis]
- india_msw_avg_kg_per_person_day: 0.57  [NITI Aayog — Waste-Wise Cities]


## Helper functions

`coalesce()` implements the first-non-null merge for verified duplicate/versioned column groups.
`drop_all_null()` removes columns nobody ever answered (dead form-revision fields). `bbox_filter()` /
`haversine_km()` / `assign_zone()` handle GPS sanity-checking and the nearest-zone spatial join.


In [4]:
def coalesce(df, canonical, sources, drop_sources=True):
    """First-non-null merge of `sources` (priority order) into a new `canonical` column."""
    present = [c for c in sources if c in df.columns]
    if not present:
        return df
    out = df[present[0]].copy()
    for c in present[1:]:
        out = out.where(out.notna(), df[c])
    df[canonical] = out
    if drop_sources:
        df = df.drop(columns=[c for c in present if c != canonical])
    return df


def drop_all_null(df, label):
    null_cols = [c for c in df.columns if df[c].notna().sum() == 0]
    if null_cols:
        print(f"[{label}] dropping {len(null_cols)} all-null columns (unanswered form-revision fields):")
        for c in null_cols:
            print("   -", c)
    return df.drop(columns=null_cols)


def bbox_filter(df, lat_col, lon_col):
    lat = pd.to_numeric(df[lat_col], errors="coerce")
    lon = pd.to_numeric(df[lon_col], errors="coerce")
    ok = lat.between(NAINITAL_BBOX["lat_min"], NAINITAL_BBOX["lat_max"]) & \
         lon.between(NAINITAL_BBOX["lon_min"], NAINITAL_BBOX["lon_max"])
    n_bad = int((~ok).sum() - df[lat_col].isna().sum())
    return ok, n_bad


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def assign_zone(df, lat_col, lon_col, zones_df, max_km=0.5):
    zone_ids, dists = [], []
    for lat, lon in zip(df[lat_col], df[lon_col]):
        if pd.isna(lat) or pd.isna(lon):
            zone_ids.append(None); dists.append(None); continue
        d = haversine_km(lat, lon, zones_df["latitude"].values, zones_df["longitude"].values)
        i = np.argmin(d)
        if d[i] <= max_km:
            zone_ids.append(zones_df["zone_id"].iloc[i]); dists.append(round(float(d[i]), 3))
        else:
            zone_ids.append(None); dists.append(round(float(d[i]), 3))
    df["zone_id"] = zone_ids
    df["zone_dist_km"] = dists
    return df


def missingness_plot(df, title, top_n=25):
    miss = df.isna().mean().sort_values(ascending=False).head(top_n)
    fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(miss))))
    ax.barh(miss.index[::-1], (miss.values * 100)[::-1], color="#4C72B0")
    ax.set_xlabel("% missing")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


quality_report = []  # (survey, rows_in, rows_out, cols_in, cols_out, gps_outliers)


---
## 1. Enterprise survey (businesses)


In [5]:
ent_raw = pd.read_excel(
    DATA_DIR / "Enterprise_Urban_Immersion_-_all_versions_-_labels_-_2026-07-05-06-03-02.xlsx",
    sheet_name="Enterprise_Urban Immersion",
)
print("shape:", ent_raw.shape)
ent_raw.head(3)


shape: (157, 66)


,Do you give consent to be part of the Survey?,Record your current location,_Record your current location_latitude,_Record your current location_longitude,_Record your current location_altitude,_Record your current location_precision,Type of Enterprise,"If other, explain",Is the owner of this business a resident of Nainital District?,How many full-time employees are currently employed in your establishment?,How many seasonal or temporary employees do you hire during peak tourist seasons?,What is the number of employees that are residents of Nainital District?,Are exclusive local handicrafts/artifacts available ?,Does your enterprise have dustbins for waste collection?,Does your enterprise have dustbins for waste collection? /Yes,Does your enterprise have dustbins for waste collection? /No,"During tourist season, as the waste generation increases, how do you manage it?",What's the primary source of your establishment's water supply?,Source of Water,What is your average monthly electricity bill during peak season (in rupees) ?,What is your average monthly electricity bill during off season (in rupees) ?,Does your establishment use solar panels or any other renewable energy source for water heating or electricity?,Do you observe presence of Child Labour in the Enterprise ?,Do you actively promote Kumaoni culture to tourists?,How many beds does your hotel have?,What is the maximum number of guests your hotel can accommodate at one time? (Including Extra Beds),Do you actively promote Kumaoni cuisine to tourists?,What is the storage tanks capacity? (in litres),"During peak season, have water constraints affected your ability to operate at full capacity?",Explain the adjustment you make during water constraints?,...,Yearly Water Expenses,How many hours a day do you receive water supply?,Is hours of water supply different during peak season?,Does your establishment have a backup diesel generator?,"Do you receive any government or institutional support (subsidies, welfare, training) during the off-season?",What type of tourism-related business do you operate?,"Are exclusive local handicrafts/artifacts available? If yes, what type?","What cultural elements do you promote? (Festivals, heritage sites, folk traditions)",Does your hotel separate dry and wet waste?,Does your hotel separate dry and wet waste? /Yes,Does your hotel separate dry and wet waste? /No,"If yes then, what channels do you use?","If yes then, what channels do you use? /Social Media","If yes then, what channels do you use? /Websites","If yes then, what channels do you use? /Banners & Brochures","If yes then, what channels do you use? /Others",What’s the frequency of the waste collection during Peak Season?,What’s the frequency of the waste collection during Off Season?,"If others, explain",Does your enterprise have a rainwater harvesting system?,How many water storage tanks does your establishment have?,Is there water shortage during peak tourist season?,How much water is consumed in a day (in litres) ?,How much water can it store roughly? (in litres),Would you be open to installing or improving a rainwater system if you do not have one?,Does your hotel reuse water from sinks or laundry in any way?,How many days in a month do you face water shortage?,Is water shortage worse during peak season?,Unnamed: 64,Total - 64
0,Yes,NaN,NaN,NaN,NaN,NaN,Travel Agency,NaN,Yes,2.0,1.0,3.0,NaN,Yes,1.0,0.0,NaN,Municipal Water Supply,NaN,3000.0,3000.0,No,NaN,NaN,NaN,NaN,NaN,NaN,Not at all,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Social Media Banners & Brochures,1.0,0.0,1.0,0.0,NaN,NaN,NaN,No,NaN,No,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.0
1,Yes,NaN,NaN,NaN,NaN,NaN,Other,Parlour/Saloon,Yes,3.0,0.0,3.0,No,Yes,1.0,0.0,No hinderence as Municipal waste truck comes 3...,Municipal Water Supply,NaN,1500.0,1000.0,No,NaN,No,NaN,NaN,No,0.0,Minor adjustments,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Others,0.0,0.0,0.0,1.0,NaN,NaN,NaN,No,NaN,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,31.0
2,Yes,NaN

In [6]:
missingness_plot(ent_raw, "Enterprise — missingness by column (top 25)")


/var/folders/9_/bt22y6vx03z64t_v4_xypn9h0000gn/T/ipykernel_32615/1824992966.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
ent = ent_raw.rename(columns={
    "_Record your current location_latitude": "latitude",
    "_Record your current location_longitude": "longitude",
})
n_rows_in, n_cols_in = ent.shape

ok, n_bad_gps = bbox_filter(ent, "latitude", "longitude")
print(f"GPS points outside Nainital bounding box: {n_bad_gps} (nulled, row kept — survey answers still valid)")
ent.loc[~ok, ["latitude", "longitude"]] = np.nan

ent = drop_all_null(ent, "ENTERPRISE")

quality_report.append(["Enterprise", n_rows_in, len(ent), n_cols_in, ent.shape[1], n_bad_gps])
print("\nfinal shape:", ent.shape)


GPS points outside Nainital bounding box: 9 (nulled, row kept — survey answers still valid)
[ENTERPRISE] dropping 1 all-null columns (unanswered form-revision fields):
   - Unnamed: 64

final shape: (157, 65)


In [8]:
fig, ax = plt.subplots(figsize=(7, 4))
ent["Type of Enterprise"].value_counts(dropna=False).plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_title("Enterprise — type distribution")
plt.tight_layout()
plt.show()

print(ent["Does your enterprise have dustbins for waste collection? "].value_counts(dropna=False))
print()
print(ent["How does your establishment manage sewage? "].value_counts(dropna=False))


Does your enterprise have dustbins for waste collection? 
Yes       135
No         16
NaN         3
Yes No      2
2           1
Name: count, dtype: int64

How does your establishment manage sewage? 
NaN                                                       75
Connected to municipal sewage line                        66
No system in place                                         9
Septic tank                                                3
Connected to municipal sewage line No system in place      1
Connected to municipal sewage line Septic tank             1
Septic tank No system in place                             1
74                                                         1
Name: count, dtype: int64


/var/folders/9_/bt22y6vx03z64t_v4_xypn9h0000gn/T/ipykernel_32615/2773954470.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Note**: several Enterprise numeric fields (e.g. `How many days in a month do you face water
shortage?`, `How many hours a day do you receive water supply?`) have only **1 non-null response**
each — a late form-revision question almost nobody was asked. Kept in the cleaned table (it's real
data, not corrupted), but flagged here: **don't build a zone-level or town-wide water-shortage-days
metric from this single response** — it will not be representative. `VARIABLES_AND_DATA_SOURCES.md`
should note this as a known-thin variable; a more robust proxy is
`water_constraint_limits_ops` / `water_shortage_worse_peak` (much higher response counts).


In [9]:
thin_cols = ["How many days in a month do you face water shortage?",
             "How many hours a day do you receive water supply? "]
for c in thin_cols:
    if c in ent.columns:
        print(f"{c!r}: {ent[c].notna().sum()} non-null / {len(ent)} rows")


'How many days in a month do you face water shortage?': 1 non-null / 157 rows
'How many hours a day do you receive water supply? ': 1 non-null / 157 rows


---
## 2. Location survey (10 site audits — our zone set)


In [10]:
loc_raw = pd.read_excel(DATA_DIR / "Location_Urban_Immersion_-_all_versions_-_labels_-_2026-07-05-06-04-27.xlsx")
print("shape:", loc_raw.shape)
loc_raw.head(3)


shape: (10, 83)


,start,end,Geolocation,_Geolocation_latitude,_Geolocation_longitude,_Geolocation_altitude,_Geolocation_precision,Time taken by ambulance/fire services to reach critical points in emergency (mins),Availability of shared transport services- Which types of shared or public transport are available near this site? (Select all that apply),Availability of shared transport services- Which types of shared or public transport are available near this site? (Select all that apply) /E-rickshaw,Availability of shared transport services- Which types of shared or public transport are available near this site? (Select all that apply) / Auto-rickshaw,Availability of shared transport services- Which types of shared or public transport are available near this site? (Select all that apply) / Cab or taxi stand,Number of hotel based parking spots,Number of public parking spots,Are there dedicated footpaths?,"If not, is there any other marking for pedestrians to walk?",Number of street lights,Is the tourism infrastructure disable friendly?,Are there proper markings/lines on the roads,Are there any potholes on the road,Number of Crossing Paths (Zebra Crossing),Is there water logging on the road?,How many directional signboards are present at the site?,How many multilingual signboards are present?,Are tourist rules and regulations displayed at the site?,Are environmental guidelines displayed?,Emergency contact information displayed (Police/Hospital/Fire Station),Is disaster/evacuation information displayed?,Number of Tourist Information Centres_Public,Number of Tourist Information Centres_Private,...,"If yes, what is the width of the footpath","If yes, what is the lengthof the footpath",Number of streetligths,Availability of accessible infrastructure at major tourist destinations fulfill the right to barrier-free tourism for persons with disabilities?,Is there an adequate number of speedbreakers on the road?,Is there proper drainage system on the road?,Are entry fees and timings displayed?,How visible is regulatory information at the site?,Emergency contact information displayed (Yes/No),Is disaster/evacuation information available?,How many operational tourist information centres are present,Are tourist rules and regulations displayed at the site?.1,Is 4G or 5G mobile network available at key tourist nodes and trekking routes? (Yes / Partial / No — note operator),What is the Total number of dustbins in the zone,"""Whar is the type of dustbin: a) Single bin b) Dry/Wet segregation bins c) Other""",Was it cleaned?,Availability of soap,Was there proper water flow?,Total number of registered accommodation units operating within the area.,_id,_uuid,_submission_time,_validation_status,_notes,_status,_submitted_by,__version__,_tags,meta/rootUuid,_index
0,2026-06-25 09:20:30.280,2026-06-25 14:30:37.140,29.390786 79.455892 1946.3156783896025 43.2007...,29.390786,79.455892,1946.315678,43.200797,10.0,E-rickshaw,1.0,0.0,0.0,3.0,2.0,Yes,NaN,3,No,No,Yes,No,No,1,0,Yes,Yes,No,No,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,791799825,891581eb-9022-417f-af5b-aa211f195c79,2026-06-25 09:00:39,NaN,NaN,submitted_via_web,NaN,v9odu7n7ing9dr7VCjQ67g,NaN,uuid:891581eb-9022-417f-af5b-aa211f195c79,1
1,2026-06-25 14:30:37.175,2026-06-25 14:47:44.302,29.390439 79.455517 1945.1488430325635 51.0564...,29.390439,79.455517,1945.148843,51.056421,NaN,Cab or taxi stand,0.0,0.0,1.0,0.0,1.0,Yes,NaN,3,No,No,No,No,No,1,0,Yes,Yes,No,No,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,791824951,6247bb18-2e79-4dae-92cd-d419d5804a53,2026-06-25 09:17:46,NaN,NaN,submitted_via_web,NaN,v9odu7n7ing9dr7VCjQ67g,NaN,uuid:6247bb18-2e79-4dae-92cd-d419d5804a53,2
2,2026-06-25 09:28:57.239,2026-06-25 15:42:13.915,29.390254 79.455142 1900.800048828125 18.57999...,29.390254,79.455142,1900.800049,18.580000,NaN,NaN,NaN,NaN,NaN,0.0,1.0,Yes,NaN,17,No,NaN,NaN,NaN,NaN,0,2,Yes,No,No,No,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [11]:
loc = loc_raw.rename(columns={"_Geolocation_latitude": "latitude", "_Geolocation_longitude": "longitude"})
n_rows_in, n_cols_in = loc.shape

# Verified duplicate groups (same question re-asked under a "New Question\"" prefixed label,
# or a trailing ".1" from a later form revision):
loc = coalesce(loc, "shared_transport_modes", [
    "Availability of shared transport services- Which types of shared or public transport are available near this site? (Select all that apply) ",
    "New Question\"Availability of shared transport services- Which types of shared or public transport are available near this site? (Select all that apply) ",
])
loc = coalesce(loc, "tourist_rules_displayed", [
    "Are tourist rules and regulations displayed at the site?",
    "Are tourist rules and regulations displayed at the site?.1",
])

loc = drop_all_null(loc, "LOCATION")
loc["zone_id"] = [f"Z{i+1:02d}" for i in range(len(loc))]

quality_report.append(["Location", n_rows_in, len(loc), n_cols_in, loc.shape[1], 0])
loc[["zone_id", "latitude", "longitude"]]


[LOCATION] dropping 26 all-null columns (unanswered form-revision fields):
   - New Question"Availability of shared transport services- Which types of shared or public transport are available near this site? (Select all that apply) /E-rickshaw
   - New Question"Availability of shared transport services- Which types of shared or public transport are available near this site? (Select all that apply) / Auto-rickshaw
   - New Question"Availability of shared transport services- Which types of shared or public transport are available near this site? (Select all that apply) / Cab or taxi stand
   - How many parking spots (hotel + public) are available in the zone?
   -  If yes, what is the width of the footpath
   -  If yes, what is the lengthof the footpath
   - Number of streetligths
   - Availability of accessible infrastructure at major tourist destinations fulfill the right to barrier-free tourism for persons with disabilities?
   - Is there an adequate number of speedbreakers on the roa

,zone_id,latitude,longitude
0,Z01,29.390786,79.455892
1,Z02,29.390439,79.455517
2,Z03,29.390254,79.455142
3,Z04,29.391620,79.456049
4,Z05,29.384868,79.463202
5,Z06,29.386094,79.457074
6,Z07,29.388482,79.460324
7,Z08,29.380271,79.463022
8,Z09,29.388446,79.460493
9,Z10,29.384327,79.463210


Only 10 zones — this is our fixed lookup table (no statistical modeling needed on n=10, just
carry it through as-is). **Zone names still need to be filled in** (reverse-geocode via OSM
Nominatim per `workflow.md` §3, or supplied manually) — placeholder `Z01`..`Z10` IDs used for now.


In [12]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(loc["longitude"], loc["latitude"], s=80, c="#C44E52")
for _, r in loc.iterrows():
    ax.annotate(r["zone_id"], (r["longitude"], r["latitude"]), textcoords="offset points", xytext=(5, 5))
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("10 surveyed zones — Nainital")
plt.tight_layout()
plt.show()


/var/folders/9_/bt22y6vx03z64t_v4_xypn9h0000gn/T/ipykernel_32615/3326308750.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 3. Tourist survey

Kobo splits each Likert question into a **label column** (always empty — it's a note widget) and
the **actual numeric answer** in the column right after it, named after the scale legend (e.g.
`"(1 Very Difficult – 5 Very Easy)"`). Verified by checking non-null counts on both columns before
coalescing — this is *not* a blind-regex guess.


In [13]:
tour_raw = pd.read_excel(DATA_DIR / "Tourist_Urban_Immersion_-_all_versions_-_labels_-_2026-07-05-06-03-51.xlsx")
print("shape:", tour_raw.shape)
tour_raw.head(3)


shape: (320, 99)


,start,end,"This survey is done as part of the Urban Immersion program of BS Analytics and Sustainability Studies, 2024-28 Batch of Tata Institute of Social Sciences. This is part of the educational program and the data collected is strictly anonymous and the responses are not be used for any commercial purpose.",Do you give consent to be part of the survey?,Gender of the respondent?,Is this your first visit to Nainital?,What is your age group?,How many people are you travelling with?,Which state/country are you visiting from?,What is the main purpose of your visit?,What is the main purpose of your visit?/Leisure/Vacation,What is the main purpose of your visit?/Vacation,What is the main purpose of your visit?/Honeymoon,What is the main purpose of your visit?/Family Trip,What is the main purpose of your visit?/Pilgrimage,What is the main purpose of your visit?/Adventure,What is the main purpose of your visit?/Business+Leisure,What is the main purpose of your visit?/Other,Length of your stay,What type of accommodation are you staying in?,Approximate total expenditure during this visit (excluding travel to Nainital),Have you gone beyond your expected budget for the trip?,Reason for overspending,How easy was it to find directions to tourist attractions?,(1 Very Difficult – 5 Very Easy),Which of the following information did you notice at tourist locations?,Which of the following information did you notice at tourist locations?/ Entry fees & timings,Which of the following information did you notice at tourist locations?/Environmental guidelines,Which of the following information did you notice at tourist locations?/Emergency contact info,Which of the following information did you notice at tourist locations?/Local rules & regulations,...,How did you travel to Nainital?,How did you travel to Nainital?/Own Vehicle,How did you travel to Nainital?/Taxi,How did you travel to Nainital?/Train,How did you travel to Nainital?/Bus,Which of the following information did you notice at tourist locations?.1,Which of the following information did you notice at tourist locations?/Emergency contact info.1,Which of the following information did you notice at tourist locations?/Local rules & regulations .1,Which of the following information did you notice at tourist locations?/ Entry fees & timings .1,Which of the following information did you notice at tourist locations?/Environmental guidelines .1,Which of the following information did you notice at tourist locations?/None.1,How many people are travelling with you in your group?,How many nights are you staying in Nainital?,"Approximately how much have you spent or expect to spend during your visit to Naini Lake today, including food, boating, shopping, and transport?",Enter your question.2,How would you rate the cleanliness of the areas you have visited in Nainital?.1,(1 Very Dissatisfied – 5 Very Satisfied).3,Enter your question.3,Did you use any of these public amenities? Public toilets / Drinking water / Seating/rest areas / Dustbins,_id,_uuid,_submission_time,_validation_status,_notes,_status,_submitted_by,__version__,_tags,meta/rootUuid,_index
0,2026-06-25 09:22:06.698,2026-06-25 09:31:30.811,NaN,NaN,NaN,No,26–35,5.0,Delhi,Family Trip,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN,Hotel,"More than ₹20,000",NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,7.0,800.0,NaN,NaN,NaN,NaN,Yes,791534393,aac25174-bbb8-4c23-aedb-4874189ede08,2026-06-25 04:01:32,NaN,NaN,submitted_via_web,NaN,vEC6K3NyxrTNrYGRknsAEw,NaN,uuid:aac25174-bbb8-4c23-aedb-4874189ede08,1
1,2026-06-25 09:38:32.107,2026-06-25 09:45:59.440,NaN,Yes,Female,No,Above 60,42.0,Maharashtra,Vacation Family Trip,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN,Guest House,"More than ₹20,000",NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,...,Taxi Train,0.0,1.0,1.0,0.0,Entry fees & timings Environmental guidelines,0.0,0.0,1.0,1.0,0.0,NaN,4.0,NaN,NaN,NaN,NaN,NaN,Yes,791540365,9f3d4189-6fb9-45eb-95b4-345bfec1224a,2026-06-25 04:16:01,NaN,NaN,su

In [14]:
tour = tour_raw.copy()
n_rows_in, n_cols_in = tour.shape

tour = coalesce(tour, "ease_of_navigation_1to5", ["(1 Very Difficult – 5 Very Easy)"])
tour = coalesce(tour, "signage_sufficiency_1to5", ["(1 Very Dissatisfied – 5 Very Satisfied)"])
tour = coalesce(tour, "cleanliness_rating_1to5", ["(1 Very Dissatisfied – 5 Very Satisfied).1"])
tour = coalesce(tour, "overall_satisfaction_1to5", ["(1 Very Dissatisfied – 5 Very Satisfied).2"])

tour = drop_all_null(tour, "TOURIST")  # drops the never-answered later-revision columns (Naini Lake spend, etc.)

quality_report.append(["Tourist", n_rows_in, len(tour), n_cols_in, tour.shape[1], 0])
print("\nfinal shape:", tour.shape)
print("overall_satisfaction_1to5 non-null:", tour["overall_satisfaction_1to5"].notna().sum(), "/", len(tour))


[TOURIST] dropping 17 all-null columns (unanswered form-revision fields):
   - This survey is done as part of the Urban Immersion program of BS Analytics and Sustainability Studies, 2024-28 Batch of Tata Institute of Social Sciences. This is part of the educational program and the data collected is strictly anonymous and the responses are not be used for any commercial purpose.   
   - How easy was it to find directions to tourist attractions?
   - Were signboards and information boards useful/sufficient? 
   - How would you rate the cleanliness of the areas you have visited in Nainital?
   - Overall, how satisfied are you with your experience in Nainital?
   - Enter your question
   - How easy was it to find directions to tourist attractions?.1
   - Enter your question.1
   - Were signboards and information boards useful/sufficient? 
   - Enter your question.2
   - How would you rate the cleanliness of the areas you have visited in Nainital?.1
   - (1 Very Dissatisfied – 5 Very Satisf

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, title in zip(
    axes,
    ["ease_of_navigation_1to5", "cleanliness_rating_1to5", "overall_satisfaction_1to5"],
    ["Ease of navigation", "Cleanliness rating", "Overall satisfaction"],
):
    tour[col].value_counts().sort_index().plot(kind="bar", ax=ax, color="#55A868")
    ax.set_title(title); ax.set_xlabel("1 (worst) – 5 (best)")
plt.tight_layout()
plt.show()

tour["What is your age group?"].value_counts(dropna=False).plot(kind="barh", figsize=(6, 3), color="#4C72B0", title="Tourist age group")
plt.tight_layout()
plt.show()


/var/folders/9_/bt22y6vx03z64t_v4_xypn9h0000gn/T/ipykernel_32615/1620249751.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/9_/bt22y6vx03z64t_v4_xypn9h0000gn/T/ipykernel_32615/1620249751.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Local Residents survey


In [16]:
res_raw = pd.read_excel(DATA_DIR / "Urban_Immersion-Local_Residents_-_all_versions_-_labels_-_2026-07-05-06-03-20.xlsx")
print("shape:", res_raw.shape)
res_raw.head(3)


shape: (239, 91)


,start,end,"This survey is done as part of the Urban Immersion program of BS Analytics and Sustainability Studies, 2024-28 Batch of Tata Institute of Social Sciences. This is part of the educational program and the data collected is strictly anonymous and the responses are not be used for any commercial purpose.",Do you give consent to be part of the data collection?,Location,_Location_latitude,_Location_longitude,_Location_altitude,_Location_precision,"How many people live in your household, including children and elderly members?",Where does your household primarily get its drinking water from?,Where does your household primarily get its drinking water from? /Municipal tap,Where does your household primarily get its drinking water from? /Bore-well,Where does your household primarily get its drinking water from? /Tanker supply,Where does your household primarily get its drinking water from? /Natural spring,Where does your household primarily get its drinking water from? /More than one source — please specify,Where does your household primarily get its drinking water from? /Others,Where does your household primarily get its drinking water from? /Municipal tap .1,If others Specify,How many hours per day does your household typically receive piped water supply?,What is the Sewage System of your house?,"If Others, specify",Do you experience any sewage overflow or sanitation problems during the peak tourist season?,How has water availability changed in your locality over the past five years?,How does your household cope during periods of water shortage?,How does your household cope during periods of water shortage?/Water Tankers,How does your household cope during periods of water shortage?/Fetch from spring,How does your household cope during periods of water shortage?/Reduce Usage,How does your household cope during periods of water shortage?/Stored in advance,How does your household cope during periods of water shortage?/Others,...,"In your experience, how do tourists in Nainital generally conduct themselves with respect to local customs and traditions?",Do you think that tourism has helped Nainital to grow and develop?,"If Yes, what are the positive aspects","If No, What are the problems",More than one source — please specify,"If yes, would you say water availability has improved, worsened, or stayed the same?",Have you noticed any change in water availability in your area over the past five years?,"If yes, would you say water availability has",Does your household use low-flow or water-saving taps and showers?,Does your household use low-flow or water-saving taps and showers? /Low-flow or water-saving taps,Does your household use low-flow or water-saving taps and showers? /Showers,"Is tourism your primary source of livelihood, or is it supplementary income?",Others.2,Rate-,Rate,Are you currently employed in tourism sector/activities ?,"If yes, how many months, do you work in tourism?",Rate Noise Pollution,How would you describe the noise levels in your neighbourhood during the tourist season compared to the rest of the year? .1,_id,_uuid,_submission_time,_validation_status,_notes,_status,_submitted_by,__version__,_tags,meta/rootUuid,_index
0,2026-06-26 12:05:30.905,2026-06-26 12:12:38.656,NaN,Yes,29.390424 79.455256 1944.8444878426503 38.7837...,29.390424,79.455256,1944.844488,38.783771,NaN,Municipal tap,1,NaN,0,0,NaN,0,NaN,NaN,4.0,Sewage Connection,NaN,Yes,NaN,Water Tankers Fetch from spring Reduce Usage ...,1.0,1.0,1.0,1.0,0.0,...,"Mixed — some respectful, some not",Yes,Development,NaN,NaN,NaN,Yes,Worsened,Showers,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Much louder — it disrupts my daily routine,792877737,5d7457db-d1bf-4adb-9cef-93ae436a3e02,2026-06-26 06:42:40,NaN,NaN,submitted_via_web,NaN,vkRuc6fqr8r95hQ9MbqtuY,NaN,uuid:5d7457db-d1bf-4adb-9cef-93ae436a3e02,1
1,2026-06-26 12:05:21.592,2026-06-26 12:13:10.346,NaN,Yes,NaN,NaN,NaN,NaN,NaN,NaN,Municipal tap,1,NaN,0,0,NaN,0,NaN,NaN,4.0,Sewage Connection,NaN,Yes,NaN,Fetch from spring

In [17]:
res = res_raw.rename(columns={"_Location_latitude": "latitude", "_Location_longitude": "longitude"})
n_rows_in, n_cols_in = res.shape

ok, n_bad_gps = bbox_filter(res, "latitude", "longitude")
print(f"GPS points outside Nainital bounding box: {n_bad_gps} (nulled, row kept)")
res.loc[~ok, ["latitude", "longitude"]] = np.nan

# Verified duplicate/reworded groups: dominant column (main form version) first, minor/later-revision
# column as fallback — checked their non-null counts are disjoint-ish before merging.
res = coalesce(res, "water_availability_trend_5yr", [
    "How has water availability changed in your locality over the past five years?",
    "Have you noticed any change in water availability in your area over the past five years? ",
])
res = coalesce(res, "noise_rating_1to5", [
    "How would you describe the noise levels in your neighbourhood during the tourist season compared to the rest of the year? ",
])
res = coalesce(res, "tourism_primary_livelihood", [
    "Is tourism  primary source of livelihood for your family?",
    "Is tourism your primary source of livelihood, or is it supplementary income?",
])

res = drop_all_null(res, "RESIDENTS")

quality_report.append(["Residents", n_rows_in, len(res), n_cols_in, res.shape[1], n_bad_gps])
print("\nfinal shape:", res.shape)


GPS points outside Nainital bounding box: 19 (nulled, row kept)
[RESIDENTS] dropping 14 all-null columns (unanswered form-revision fields):
   - This survey is done as part of the Urban Immersion program of BS Analytics and Sustainability Studies, 2024-28 Batch of Tata Institute of Social Sciences. This is part of the educational program and the data collected is strictly anonymous and the responses are not be used for any commercial purpose.   
   - Rate- Traveling in Nainital
   - More than one source — please specify
   - If yes, would you say water availability has improved, worsened, or stayed the same? 
   - Others.2
   - Rate-
   - Rate
   - Are you currently employed in tourism sector/activities ?
   - If yes, how many months, do you work in tourism?
   - Rate Noise Pollution
   - _validation_status
   - _notes
   - _submitted_by
   - _tags

final shape: (239, 75)


In [18]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
res["Where does your household primarily get its drinking water from?  "].value_counts(dropna=False).plot(
    kind="barh", ax=axes[0], color="#4C72B0", title="Household primary water source")
res["water_availability_trend_5yr"].value_counts(dropna=False).plot(
    kind="barh", ax=axes[1], color="#C44E52", title="Water availability trend, 5yr")
plt.tight_layout()
plt.show()

print(res["tourism_primary_livelihood"].value_counts(dropna=False))


tourism_primary_livelihood
No     121
Yes    117
NaN      1
Name: count, dtype: int64


/var/folders/9_/bt22y6vx03z64t_v4_xypn9h0000gn/T/ipykernel_32615/3433880309.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Workers survey


In [19]:
work_raw = pd.read_excel(DATA_DIR / "Workers_Urban_Immersion_-_all_versions_-_labels_-_2026-07-05-06-03-35.xlsx")
print("shape:", work_raw.shape)
work_raw.head(3)


shape: (303, 43)


,start,end,"This survey is done as part of the Urban Immersion program of BS Analytics and Sustainability Studies, 2024-28 Batch of Tata Institute of Social Sciences. This is part of the educational program and the data collected is strictly anonymous and the responses are not be used for any commercial purpose.",Do we have your consent for the study?,What is your Age?,What is your gender?,Domicile Status,What is your highest level of education?,What sector do you work in?,If Other (specify),Status of employment,How many years have you been working in this occupation?,How many hours do you work in a day?,Have you received any training for employment?,How would you describe your current employment arrangement?,For how many months in the previous year you received work?,What is your approximate monthly income during peak tourist season?,What is your approximate monthly income during the off-season?,Approximately what share of your total annual income comes from tourism-related activities?,Do you have any other source of income aside from tourism-linked work?,If Yes (specify),Do you have access to any health insurance or accident coverage related to your work?,What are the opportunities that tourism provides?,Other (specify),Is tourism your primary source of employment?,Do you receive any seasonal training or briefing before peak tourist season?,Yes (specify),"Do you receive any government or institutional support (subsidies, welfare, training)?",Yes (specify).1,Do you receive your payment on time?,What are the challenges you face due to tourism in your employment?,Which challenge affects your tourism income the most?,_id,_uuid,_submission_time,_validation_status,_notes,_status,_submitted_by,__version__,_tags,meta/rootUuid,_index
0,2026-06-25 09:18:31.589,2026-06-25 09:21:27.425,NaN,Yes,45.0,Male,Migrant,Graduate & Above,Street Vendors,NaN,NaN,22.0,7.0,NaN,Seasonal,12.0,2.500000e+04,15000.0,100%,No,NaN,No,No opportunities,NaN,NaN,No,NaN,NaN,NaN,NaN,"Less money in off season, standard of living c...",NaN,791530915,5fe47c4c-52e5-42fc-b526-419656c1aead,2026-06-25 03:51:28,NaN,NaN,submitted_via_web,NaN,vkogsw88uLkMNrsaYGUAqr,NaN,uuid:5fe47c4c-52e5-42fc-b526-419656c1aead,1
1,2026-06-25 09:19:21.548,2026-06-25 09:39:25.674,NaN,Yes,30.0,Male,Native,Graduate & Above,"Transport (Taxi, Bike, Autorickshaw)",NaN,NaN,9.0,10.0,NaN,Seasonal,6.0,3.500000e+04,10000.0,More than 75%,No,NaN,Yes,Theek hai,NaN,NaN,No,NaN,NaN,NaN,NaN,"Traffic, food, dhoop",NaN,791537362,41b1dec2-a7b5-40ea-b939-79a2da3da283,2026-06-25 04:09:27,NaN,NaN,submitted_via_web,NaN,vh5af8k4pthJ2kjN2ewzGa,NaN,uuid:41b1dec2-a7b5-40ea-b939-79a2da3da283,2
2,2026-06-25 09:50:39.080,2026-06-25 09:59:53.884,NaN,Yes,23.0,Male,Native,Graduate & Above,"Transport (Taxi, Bike, Autorickshaw)",NaN,NaN,2.0,7.0,NaN,Seasonal,12.0,4.000060e+09,1000.0,100%,No,NaN,No,Taxi driver,NaN,NaN,No,NaN,NaN,NaN,NaN,"Chalan, medical accident",NaN,791548103,26a08704-235e-4745-830d-d0b8b931a2e5,2026-06-25 04:29:54,NaN,NaN,submitted_via_web,NaN,vh5af8k4pthJ2kjN2ewzGa,NaN,uuid:26a08704-235e-4745-830d-d0b8b931a2e5,3


In [20]:
work = work_raw.copy()
n_rows_in, n_cols_in = work.shape
work = drop_all_null(work, "WORKERS")
quality_report.append(["Workers", n_rows_in, len(work), n_cols_in, work.shape[1], 0])
print("\nfinal shape:", work.shape)


[WORKERS] dropping 12 all-null columns (unanswered form-revision fields):
   - This survey is done as part of the Urban Immersion program of BS Analytics and Sustainability Studies, 2024-28 Batch of Tata Institute of Social Sciences. This is part of the educational program and the data collected is strictly anonymous and the responses are not be used for any commercial purpose.   
   - Other (specify)
   - Is tourism your primary source of employment?
   - Yes (specify)
   - Do you receive any government or institutional support (subsidies, welfare, training)?
   - Yes (specify).1
   - Do you receive your payment on time?
   - Which challenge affects your tourism income the most?
   - _validation_status
   - _notes
   - _submitted_by
   - _tags

final shape: (303, 31)


In [21]:
work["worker_income_peak"] = pd.to_numeric(
    work["What is your approximate monthly income during peak tourist season?"], errors="coerce")
work["worker_income_offpeak"] = pd.to_numeric(
    work["What is your approximate monthly income during the off-season?"], errors="coerce")

fig, ax = plt.subplots(figsize=(5, 4))
work[["worker_income_peak", "worker_income_offpeak"]].dropna().plot(kind="box", ax=ax)
ax.set_title("Worker monthly income — peak vs off-season")
plt.tight_layout()
plt.show()

print(work["Approximately what share of your total annual income comes from tourism-related activities?"].value_counts(dropna=False))
print()
print(work["Do you have access to any health insurance or accident coverage related to your work?"].value_counts(dropna=False))


Approximately what share of your total annual income comes from tourism-related activities?
100%             216
NaN               30
More than 75%     19
51-75%            16
25-50%            15
Less than 25%      7
Name: count, dtype: int64

Do you have access to any health insurance or accident coverage related to your work?
No     227
Yes     67
NaN      9
Name: count, dtype: int64


/var/folders/9_/bt22y6vx03z64t_v4_xypn9h0000gn/T/ipykernel_32615/2124525694.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 6. Zone assignment (GPS → nearest of 10 surveyed zones)


In [22]:
ent = assign_zone(ent, "latitude", "longitude", loc, max_km=0.5)
res = assign_zone(res, "latitude", "longitude", loc, max_km=0.5)

print(f"Enterprise: {ent['zone_id'].notna().sum()} / {len(ent)} records assigned to a zone (within 500m)")
print(f"Residents:  {res['zone_id'].notna().sum()} / {len(res)} records assigned to a zone (within 500m)")
print()
print("Enterprise per zone:\n", ent["zone_id"].value_counts(dropna=False).sort_index())


Enterprise: 61 / 157 records assigned to a zone (within 500m)
Residents:  143 / 239 records assigned to a zone (within 500m)

Enterprise per zone:
 zone_id
Z01     5
Z02     4
Z03     5
Z04    18
Z05     4
Z07     2
Z08    20
Z09     1
Z10     2
NaN    96
Name: count, dtype: int64


In [23]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(ent["longitude"], ent["latitude"], s=15, alpha=0.5, label="Enterprise", c="#4C72B0")
ax.scatter(res["longitude"], res["latitude"], s=15, alpha=0.5, label="Residents", c="#55A868")
ax.scatter(loc["longitude"], loc["latitude"], s=120, c="#C44E52", marker="X", label="Zone (site audit)")
for _, r in loc.iterrows():
    ax.annotate(r["zone_id"], (r["longitude"], r["latitude"]), textcoords="offset points", xytext=(5, 5))
ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title("Respondent GPS points vs. the 10 surveyed zones")
ax.legend()
plt.tight_layout()
plt.show()


/var/folders/9_/bt22y6vx03z64t_v4_xypn9h0000gn/T/ipykernel_32615/1319699018.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Reading this**: a large share of Enterprise/Resident records fall outside the 500m cutoff
(shown as `zone_id = None`, points scattered on the map above away from any red X). That's expected
— the 10 zone audits only cover specific high-footfall points (Mall Road, Naini Lake, etc.), not
every street in town. Those unassigned records still count fully in the **town-wide** rollup; they
just don't contribute to a specific zone's map score. If per-zone coverage needs to be denser, that's
a future data-collection note, not something to fix by forcing a bad spatial match now.


---
## 7. Benchmark sanity checks

Only the benchmarks with a verified, directly-applicable number (see `RESEARCH_BENCHMARKS.md` §1)
are checked against data here.


In [24]:
# Sphere: <20 users per toilet. We don't have a verified per-zone footfall count, so this is checked
# against the crudest available proxy — registered accommodation units in the zone — and labeled
# explicitly as a lower-bound/context check, not a real occupancy-based adequacy score.
toilet_col = "How many public toilets are there in the zone"
accom_col = "Total number of registered accommodation units operating within the area."

if toilet_col in loc.columns and accom_col in loc.columns:
    check = loc[["zone_id", toilet_col, accom_col]].copy()
    check.columns = ["zone_id", "toilet_count", "accommodation_units"]
    check["toilets_meet_sphere_floor_vs_accom_units"] = (
        check["accommodation_units"] / check["toilet_count"].replace(0, np.nan)
    ) <= BENCHMARKS["sphere_max_users_per_toilet"]["value"]
    print(check)
    print("\nCaveat: accommodation UNITS is not visitor headcount — this is a rough lower-bound check,")
    print("not a real adequacy score. Treat as directional only until a footfall estimate exists.")
else:
    print("Expected columns not found — skipping (see column names printed by drop_all_null above).")


Expected columns not found — skipping (see column names printed by drop_all_null above).


In [25]:
print("Nainital peak-season context (not applied to our data — no vehicle-count variable exists in")
print("these surveys — kept here purely as the sourced anchor for the future stress-test module):")
for k in ["nainital_road_capacity_vehicles", "nainital_peak_daily_vehicles", "nainital_peak_weekend_vehicles"]:
    print(f"  {k}: {BENCHMARKS[k]['value']}  — {BENCHMARKS[k]['note']}")


Nainital peak-season context (not applied to our data — no vehicle-count variable exists in
these surveys — kept here purely as the sourced anchor for the future stress-test module):
  nainital_road_capacity_vehicles: 3000  — Approx. vehicle capacity of Nainital's road network.
  nainital_peak_daily_vehicles: (3000, 4000)  — Reported daily tourist-vehicle inflow during peak season (range).
  nainital_peak_weekend_vehicles: 50000  — Reported peak-summer-weekend vehicle attempts vs ~3,000 capacity — i.e. ~16.7x capacity.


---
## 8. Data-quality summary & save cleaned tables


In [26]:
quality_df = pd.DataFrame(
    quality_report,
    columns=["survey", "rows_in", "rows_out", "cols_in", "cols_out", "gps_outliers_nulled"],
)
quality_df["rows_dropped"] = quality_df["rows_in"] - quality_df["rows_out"]
quality_df["cols_dropped"] = quality_df["cols_in"] - quality_df["cols_out"]
print(quality_df.to_string(index=False))

quality_df.to_csv(OUT_DIR / "data_quality_report.csv", index=False)


    survey  rows_in  rows_out  cols_in  cols_out  gps_outliers_nulled  rows_dropped  cols_dropped
Enterprise      157       157       66        65                    9             0             1
  Location       10        10       83        56                    0             0            27
   Tourist      320       320       99        82                    0             0            17
 Residents      239       239       91        75                   19             0            16
   Workers      303       303       43        31                    0             0            12


In [27]:
ent.to_csv(OUT_DIR / "enterprise_clean.csv", index=False)
loc.to_csv(OUT_DIR / "location_clean.csv", index=False)
tour.to_csv(OUT_DIR / "tourist_clean.csv", index=False)
res.to_csv(OUT_DIR / "residents_clean.csv", index=False)
work.to_csv(OUT_DIR / "workers_clean.csv", index=False)

print("Saved cleaned tables to", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*.csv")):
    print(" -", f.name)


Saved cleaned tables to /Users/shreyitsinha/Documents/Tiss_my/Sem IV/UFW/dashboard_vulnarability_tourism/processed
 - data_quality_report.csv
 - enterprise_clean.csv
 - location_clean.csv
 - residents_clean.csv
 - tourist_clean.csv
 - workers_clean.csv


## Open items carried forward to `workflow.md`

- **Zone naming**: `Z01`..`Z10` are placeholders — confirm/replace via OSM Nominatim reverse-geocode
  or manual local knowledge.
- **Thin Enterprise water-shortage-days/hours-of-supply fields** (n=1): don't use directly for any
  index; rely on the higher-response-count proxies instead (§1 note above).
- **No verified per-tourist waste or per-capita water-litres benchmark** for Nainital — WSPI/WSI will
  use relative (zone-vs-zone, peak-vs-offpeak) scoring only, per `RESEARCH_BENCHMARKS.md` §3.
- **~60–65% of Enterprise/Resident records unassigned to a zone** (outside the 500m cutoff) — counted
  in town-wide indices only, not in per-zone map scores.
- Next: build the derived-variable and composite-index computation on top of these cleaned tables
  (`workflow.md` §5), using `../processed/*.csv` as the input.
